In [1]:
import os
import pandas as pd
import numpy as np

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.0' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


### Constants

In [2]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
str_model = '03_pricing_lgd'

Project: 20231010-gen-xii


### Import compliance file

In [4]:
%%time

str_filename = 'df_race_gender_age.csv'
str_uri = f's3://{str_project}/ad_hoc/compliance/{str_filename}'
df_tmp = pd.read_csv(str_uri, usecols=['uniqueid', 'max_proba_race', 'race'])
df_tmp.dropna(subset=['race'], inplace=True)

# show
df_tmp

CPU times: user 640 ms, sys: 46 ms, total: 686 ms
Wall time: 4.05 s


,max_proba_race,race,uniqueid
0,0.977112,white,1.338314e+14
1,0.976314,white,1.338520e+14
2,0.982650,white,1.338624e+14
3,0.954750,black,1.338702e+14
4,0.990006,white,1.338756e+14
...,...,...,...
479337,0.982376,white,3.640873e+14
479338,0.672772,white,3.640880e+14
479339,0.926760,white,3.640882e+14
479340,0.977962,white,3.640888e+14


### Training data

In [5]:
%%time

# import training data
str_filename = 'df_train_noleaks_pre.gzip'
str_uri = f's3://{str_project}/{str_model}/02_model/model4/00_preprocessing/02_make_dfs/{str_filename}'
df_train = pd.read_parquet(str_uri, columns=['uniqueid'])
# show
df_train

CPU times: user 171 ms, sys: 15 ms, total: 186 ms
Wall time: 605 ms


,uniqueid
8202,1.337511e+14
8203,1.337511e+14
8204,1.337528e+14
8205,1.337528e+14
8206,1.337539e+14
...,...
42362,2.809269e+14
42363,2.809329e+14
42364,2.809361e+14
42365,2.809370e+14


In [7]:
%%time

# join
df_train = pd.merge(
    left=df_tmp,
    right=df_train,
    on='uniqueid',
    how='inner',
)

# show
df_train

CPU times: user 54.7 ms, sys: 3.05 ms, total: 57.8 ms
Wall time: 56.7 ms


,max_proba_race,race,uniqueid
0,0.982931,white,1.338768e+14
1,0.982931,white,1.338768e+14
2,0.988099,black,1.339011e+14
3,0.989486,hispanic,1.339054e+14
4,0.989486,hispanic,1.339054e+14
...,...,...,...
26022,0.538495,hispanic,2.808134e+14
26023,0.538495,hispanic,2.808134e+14
26024,0.646697,black,2.808200e+14
26025,0.978521,white,2.808213e+14


In [8]:
# subset - race
df_train = df_train[df_train['max_proba_race'] >= 0.70]

# show
df_train

,max_proba_race,race,uniqueid
0,0.982931,white,1.338768e+14
1,0.982931,white,1.338768e+14
2,0.988099,black,1.339011e+14
3,0.989486,hispanic,1.339054e+14
4,0.989486,hispanic,1.339054e+14
...,...,...,...
26018,0.736362,white,2.807715e+14
26019,0.832833,black,2.807903e+14
26021,0.983792,white,2.807995e+14
26025,0.978521,white,2.808213e+14


In [9]:
%%time

# import validation data
str_filename = 'df_valid_noleaks_pre.gzip'
str_uri = f's3://{str_project}/{str_model}/02_model/model4/00_preprocessing/02_make_dfs/{str_filename}'
df_valid = pd.read_parquet(str_uri, columns=['uniqueid'])
# show
df_valid

CPU times: user 80.7 ms, sys: 3.61 ms, total: 84.3 ms
Wall time: 533 ms


,uniqueid
42367,2.809478e+14
42368,2.809484e+14
42370,2.809527e+14
42369,2.809527e+14
42371,2.809582e+14
...,...
53751,3.659098e+14
53752,3.659383e+14
53753,3.659467e+14
53754,3.659534e+14


In [10]:
%%time

# join
df_valid = pd.merge(
    left=df_tmp,
    right=df_valid,
    on='uniqueid',
    how='inner',
)

# show
df_valid

CPU times: user 54 ms, sys: 2.9 ms, total: 56.9 ms
Wall time: 55.9 ms


,max_proba_race,race,uniqueid
0,0.984353,black,2.809484e+14
1,0.996538,white,2.809894e+14
2,0.996538,white,2.809894e+14
3,0.784839,white,2.810069e+14
4,0.945607,black,2.810263e+14
...,...,...,...
8398,0.766311,black,3.640512e+14
8399,0.493471,white,3.640262e+14
8400,0.633469,white,3.640333e+14
8401,0.979162,hispanic,3.640786e+14


In [11]:
# subset - race
df_valid = df_valid[df_valid['max_proba_race'] >= 0.70]

# show
df_valid

,max_proba_race,race,uniqueid
0,0.984353,black,2.809484e+14
1,0.996538,white,2.809894e+14
2,0.996538,white,2.809894e+14
3,0.784839,white,2.810069e+14
4,0.945607,black,2.810263e+14
...,...,...,...
8396,0.957532,black,3.640160e+14
8397,0.961936,white,3.640846e+14
8398,0.766311,black,3.640512e+14
8401,0.979162,hispanic,3.640786e+14


In [12]:
# make mapping dictionary
dict_map = {
    'white': 0,
    'black': 1,
    'hispanic': 2,
    'native': 3,
    'api': 4,
    'multiple': 5,
}

In [13]:
# map race
df_train['race'] = df_train['race'].map(dict_map)

# show
df_train

/tmp/ipykernel_17749/81640252.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train['race'] = df_train['race'].map(dict_map)


,max_proba_race,race,uniqueid
0,0.982931,0,1.338768e+14
1,0.982931,0,1.338768e+14
2,0.988099,1,1.339011e+14
3,0.989486,2,1.339054e+14
4,0.989486,2,1.339054e+14
...,...,...,...
26018,0.736362,0,2.807715e+14
26019,0.832833,1,2.807903e+14
26021,0.983792,0,2.807995e+14
26025,0.978521,0,2.808213e+14


In [14]:
# map race
df_valid['race'] = df_valid['race'].map(dict_map)

# show
df_valid

/tmp/ipykernel_17749/2934859194.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_valid['race'] = df_valid['race'].map(dict_map)


,max_proba_race,race,uniqueid
0,0.984353,1,2.809484e+14
1,0.996538,0,2.809894e+14
2,0.996538,0,2.809894e+14
3,0.784839,0,2.810069e+14
4,0.945607,1,2.810263e+14
...,...,...,...
8396,0.957532,1,3.640160e+14
8397,0.961936,0,3.640846e+14
8398,0.766311,1,3.640512e+14
8401,0.979162,2,3.640786e+14
